In [ ]:
# Install all required multi-modal dependencies
!pip install -q streamlit google-genai groq openai pillow requests huggingface_hub

# Colab endpoint IP address fetch
import urllib
print("Tunnel Password / Endpoint IP is: ", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

In [30]:
%%writefile app.py
import streamlit as st
import io
import requests
from PIL import Image
from groq import Groq
from google import genai
from google.genai import types
from openai import OpenAI
from huggingface_hub import InferenceClient
import base64
import time

# ----------------------------------------------------
# 1. API KEYS
# ----------------------------------------------------
groq_key = "PASTE"
gemini_key = "PASTE"
hf_key = "PASTE"
openai_key = "PASTE"

# Initialize SDK Clients
groq_client = Groq(api_key=groq_key) if groq_key and "PASTE" not in groq_key else None
gemini_client = genai.Client(api_key=gemini_key) if gemini_key and "PASTE" not in gemini_key else None
openai_client = OpenAI(api_key=openai_key) if openai_key and "PASTE" not in openai_key else None

# ----------------------------------------------------
# 2. UI INITIALIZATION & PIPELINE SELECTOR
# ----------------------------------------------------
st.set_page_config(page_title="MultiModalWebApp", page_icon="🌌", layout="wide")
st.title("🌌 Dashboard")
st.write("An application executing 9 distinct input-to-output modalities.")
st.markdown("---")

st.sidebar.header("🎛️ Pipeline Selector")
input_modality = st.sidebar.selectbox("1. Select Input Type:", ["Text", "Image", "Audio", "Video"])

# Filter valid output paths
if input_modality == "Text":
    output_options = ["Text", "Image", "Audio", "Video"]
elif input_modality == "Image":
    output_options = ["Text", "Image"]
elif input_modality == "Audio":
    output_options = ["Text", "Audio"]
elif input_modality == "Video":
    output_options = ["Text"]

output_modality = st.sidebar.selectbox("2. Select Output Type:", output_options)
st.info(f"🚀 Running Pipeline: **{input_modality} ➔ {output_modality}**")

# ----------------------------------------------------
# 3. THE 9 CORE MODALITY FUNCTIONS
# ----------------------------------------------------

# Text -> Text (Groq Llama-3.3)
def run_text_to_text(prompt):
    if not groq_client: return "Missing or invalid Groq API Key!"
    response = groq_client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="llama-3.3-70b-versatile",
    )
    return response.choices[0].message.content

# Text -> Image (OpenAI)
def run_text_to_image(prompt):
    if not openai_client:
        st.error("Missing OpenAI API Key!")
        return None
    try:
        response = openai_client.images.generate(
            model="gpt-image-2",
            prompt=prompt,
            size="1024x1024",
            quality="auto",
            n=1,
        )
        # Decode the raw base64 data returned directly by gpt-image-2
        img_b64 = response.data[0].b64_json
        img_bytes = base64.b64decode(img_b64)
        return Image.open(io.BytesIO(img_bytes))
    except Exception as e:
        st.error(f"OpenAI Image Gen Error: {e}")
        return None

# Text -> Audio (OpenAI TTS)
def run_text_to_audio(prompt):
    if not openai_client: st.error("Missing OpenAI API Key!"); return None
    response = openai_client.audio.speech.create(model="tts-1", voice="alloy", input=prompt)
    return response.content

# Text -> Video (Hugging Face Video Gen)
def run_text_to_video(prompt):
    if not hf_key or "PASTE" in hf_key: st.error("Missing Hugging Face Token!"); return None
    try:
        client = InferenceClient(api_key=hf_key)
        video_bytes = client.text_to_video(prompt, model="Wan-AI/Wan2.2-T2V-A14B")
        return video_bytes
    except Exception as e:
        st.error(f"HF Video Error: {e}")
        return None

# Image -> Text (Gemini 2.5 Flash Visual QA)
def run_image_to_text(image_input, prompt):
    if not gemini_client: return "Missing Gemini API Key!"
    response = gemini_client.models.generate_content(model="gemini-2.5-flash", contents=[image_input, prompt])
    return response.text

# Image -> Image (OpenAI production endpoint)
def run_image_to_image(image_input, prompt):
    if not openai_client:
        st.error("Missing OpenAI API Key!")
        return None
    try:
        # 1. Convert the input PIL image to binary PNG bytes
        byte_stream = io.BytesIO()
        image_input.save(byte_stream, format="PNG")
        byte_stream.seek(0)

        # 2. Package file manually into a standard tuple layout to force the correct mimetype
        file_tuple = ("input_file.png", byte_stream, "image/png")

        # 3. Request the image edit execution
        response = openai_client.images.edit(
            image=file_tuple,  # Explicitly passed with metadata
            model="gpt-image-2",
            prompt=prompt,
            n=1,
            size="1024x1024"
        )

        # 4. Extract and decode the base64 content
        img_b64 = response.data[0].b64_json
        img_bytes = base64.b64decode(img_b64)
        return Image.open(io.BytesIO(img_bytes))
    except Exception as e:
        st.error(f"OpenAI Image Variation Error: {e}")
        return None

# Audio -> Text (Groq Whisper-v3)
def run_audio_to_text(audio_bytes, filename):
    if not groq_client: return "Missing Groq API Key!"
    transcription = groq_client.audio.transcriptions.create(file=(filename, audio_bytes), model="whisper-large-v3")
    return transcription.text

# Audio -> Audio (Gemini Native Audio)
def run_audio_to_audio(audio_path_local):
    if not gemini_client or not openai_client:
        st.error("Missing Gemini or OpenAI API Key!")
        return None
    try:
        # 1. Upload file to Google File Cloud Storage
        audio_file = gemini_client.files.upload(file=audio_path_local)

        # 2. Instruct Gemini to act as a direct text generator based on the spoken command
        response = gemini_client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[
                audio_file,
                "Listen to the audio command. If the user asks you to say or generate a specific phrase, output ONLY that requested phrase. Do not add conversational intro text like 'Here is your audio:' or punctuation fluff."
            ]
        )
        generated_text = response.text.strip()

        if not generated_text:
            st.error("Gemini failed to interpret the audio file.")
            return None

        # 3. Render the processed target text to an audio file
        speech_response = openai_client.audio.speech.create(
            model="tts-1",
            voice="alloy",
            input=generated_text
        )
        return speech_response.content

    except Exception as e:
        st.error(f"Audio-to-Audio Pipeline Error: {e}")
        return None

# Video -> Text (Gemini Video Analysis)
def run_video_to_text(video_path_local, prompt):
    if not gemini_client:
        st.error("Missing Gemini API Key!")
        return None
    try:
        # 1. Upload the video to Google's File API
        st.info("Uploading video to processing server...")
        video_file = gemini_client.files.upload(file=video_path_local)

        # 2. Wait until the file is fully processed and ACTIVE
        st.info("Processing video frames... please wait a moment.")
        while video_file.state.name == "PROCESSING":
            time.sleep(2)
            # Refresh the file state data from the server
            video_file = gemini_client.files.get(name=video_file.name)

        # Check if processing failed for some reason
        if video_file.state.name == "FAILED":
            st.error("Video processing failed on Google's backend server.")
            return None

        st.success("Video is active! Running analysis...")

        # 3. Securely query Gemini now that the video file is completely ready
        response = gemini_client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[video_file, prompt]
        )
        return response.text

    except Exception as e:
        st.error(f"Video-to-Text Pipeline Error: {e}")
        return None

# ----------------------------------------------------
# 4. UI SPLIT COLUMNS LAYOUT
# ----------------------------------------------------
col1, col2 = st.columns(2)

with col1:
    st.subheader("📥 Input Capture Panel")
    user_prompt = ""
    uploaded_img = None
    uploaded_aud = None
    uploaded_vid = None

    if input_modality == "Text":
        user_prompt = st.text_area("Write Text Prompt:", "A cinematic snapshot of mountains at dusk.")
    elif input_modality == "Image":
        file_img = st.file_uploader("Upload Target Image", type=["png", "jpg", "jpeg"])
        if file_img:
            uploaded_img = Image.open(file_img)
            st.image(uploaded_img, caption="Source Input", width=300)
        user_prompt = st.text_input("Accompanying Instruction Prompt:", "Describe what you see")
    elif input_modality == "Audio":
        file_aud = st.file_uploader("Upload Target Audio File", type=["mp3", "wav"])
        if file_aud:
            uploaded_aud = file_aud.read()
            st.audio(uploaded_aud)
            with open("temp_audio.mp3", "wb") as f: f.write(uploaded_aud)
    elif input_modality == "Video":
        file_vid = st.file_uploader("Upload Target Video File", type=["mp4"])
        if file_vid:
            uploaded_vid = file_vid.read()
            st.video(uploaded_vid)
            with open("temp_video.mp4", "wb") as f: f.write(uploaded_vid)
        user_prompt = st.text_input("Video Instruction:", "Summarize this video scene by scene.")

    fire_trigger = st.button("🔥 Run Inference Pipeline")

with col2:
    st.subheader("📤 AI Output Terminal")
    if fire_trigger:
        with st.spinner("Processing through neural network pipeline..."):

            # --- TEXT INPUTS ---
            if input_modality == "Text" and output_modality == "Text":
                st.write(run_text_to_text(user_prompt))
            elif input_modality == "Text" and output_modality == "Image":
                out_img = run_text_to_image(user_prompt)
                if out_img: st.image(out_img)
            elif input_modality == "Text" and output_modality == "Audio":
                out_aud = run_text_to_audio(user_prompt)
                if out_aud: st.audio(out_aud)
            elif input_modality == "Text" and output_modality == "Video":
                out_vid_bytes = run_text_to_video(user_prompt)
                if out_vid_bytes: st.video(out_vid_bytes, format="video/mp4")

            # --- IMAGE INPUTS ---
            elif input_modality == "Image" and output_modality == "Text":
                if uploaded_img: st.write(run_image_to_text(uploaded_img, user_prompt))
                else: st.warning("Upload an image first.")
            elif input_modality == "Image" and output_modality == "Image":
                if uploaded_img:
                    out_img = run_image_to_image(uploaded_img, user_prompt)
                    if out_img: st.image(out_img)
                else: st.warning("Upload a base image first.")

            # --- AUDIO INPUTS ---
            elif input_modality == "Audio" and output_modality == "Text":
                if uploaded_aud: st.write(run_audio_to_text(uploaded_aud, file_aud.name))
                else: st.warning("Upload an audio file first.")
            elif input_modality == "Audio" and output_modality == "Audio":
                if uploaded_aud:
                    out_speech = run_audio_to_audio("temp_audio.mp3")
                    if out_speech: st.audio(out_speech)
                else: st.warning("Upload an audio file first.")

            # --- VIDEO INPUTS ---
            elif input_modality == "Video" and output_modality == "Text":
                if uploaded_vid: st.write(run_video_to_text("temp_video.mp4", user_prompt))
                else: st.warning("Upload a video file first.")

Overwriting app.py


In [ ]:
# Start Streamlit silently in the background
!streamlit run app.py &>/content/logs.txt &

# Expose the port using localtunnel
!npx localtunnel --port 8501

In [ ]:
# 1. Install pyngrok
!pip install -q pyngrok

# 2. Authenticate the ngrok agent
from pyngrok import ngrok
ngrok.set_auth_token("PASTE")

# 3. Force-kill any hanging backend ports from previous runs
!pkill streamlit
ngrok.kill()

# 4. Boot Streamlit cleanly in the background
!streamlit run app.py &>/dev/null &

# 5. Connect the ngrok tunnel directly to Streamlit's port (8501)
try:
    public_url = ngrok.connect(8501, proto="http")
    print("🚀 NGROK TUNNEL SUCCESSFUL!")
    print("👉 Click here to open your UI safely:", public_url.public_url)
except Exception as e:
    print(f"Tunnel Error: {e}")

In [ ]:
# 1. Instantly kill the background Streamlit process
!pkill streamlit

# 2. Shut down and disconnect all active ngrok tunnel connections
from pyngrok import ngrok
ngrok.kill()

print("🛑 Dashboard disconnected successfully! The public URL is now dead.")